In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
import sys
!{sys.executable} -m pip install seaborn



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
"c:\Program" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [3]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [4]:
import pandas as pd
import numpy as np
#import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
%%time
query = """
select codmes,
count(*)
from "disc_comercial"."platf_renta_alta_train_V1"
group by 1  
;"""
df_dataset = athena_query(query, database='disc_comercial')
df_dataset.head()

CPU times: total: 1.66 s
Wall time: 11.3 s


,codmes,_col1
0,202603,242426
1,202507,252464
2,202510,256789
3,202512,271339
4,202604,242291


In [6]:
%%time
query = """
select *
from "disc_comercial"."platf_renta_alta_train_V1" 
;"""
df_inference = athena_query(query, database='disc_comercial')
df_inference.head()

CPU times: total: 9min 5s
Wall time: 30min 43s


,target_m,key_value,codunico,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,...,bpi_trx_nmon_prom2,bpi_trx_mon_prom,bpi_trx_mon_prom2,bpi_trx_mon_min,bpi_trx_mon_max,bpi_recen_mon,bpi_recen_nmon,grupo_final,lvl_edu,codmes
0,0,04B1E7A87AA2C347D9790E15CC365ECA4CCAEB5B73A88F...,0001519032,202505,202506,None,6.53,900.00,850.00,1.00,...,1.00000000000000,0.75000000000000,1.80000000000000,0,5,0,0,Digital,SUPERIOR COMPLETA,202506
1,0,9447E8804F264AD30AC441B5DD36ACD105C3F227888EAA...,0009538887,202504,202505,None,0.01,NaN,NaN,0.00,...,0E-14,0E-14,0E-14,0,0,0,0,Sin transacciones,SUPERIOR COMPLETA,202505
2,0,F9DF992558D045985040597F61A133BFC2ECE2AEA755A5...,0013857209,202602,202603,None,368.52,4100.00,3020.00,1.00,...,16.00000000000000,1.00000000000000,3.00000000000000,1,6,0,0,Digital,<NA>,202603
3,0,19953A2C7B31B3EBB1E5E287169D18A743A4255BB4934B...,0011366706,202412,202501,None,315.88,170.00,26912.00,4.67,...,None,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,202501
4,0,D1A52146B5F74E0677445CAF682E3E72B5462207C7C965...,0015346677,202601,202602,None,1291.09,0.00,0.00,0.00,...,13.72727272727273,17.50000000000000,17.50000000000000,1,32,0,0,Digital,SECUNDARIA COMPLETA,202602


In [8]:
df_inference.shape

(4101854, 125)

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL   = "target_m"
PERIODO_COL  = "cod_mes"
PERIODOS     = [202501, 202502, 202503, 202504, 202505, 202506, 202507]
OUTPUT_DIR   = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarollo\bivariados_graficos"
N_BINS       = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Variables seleccionadas ──────────────────────────────────────────────────
VARIABLES_SELECCIONADAS = [
    # Transaccional / pasivo
    'mto_pas_soles',
    'imp_trx_abonosefect_6m',
    'imp_trx_cargosefe_6m',
    'avg_trx_cargostot_3m',
    'max_trx_abonos_3m',
    'cnt_trx_cargostot_3m',
    'cnt_trx_abonospromtot_3m',
    'rat_trx_abonosefectot_1m',
    'rat_trx_abonosefectot_3m',
    'rat_trx_abonosefectot_9m',
    'rat_mntcrgsefetot_1m',

    # Demográficas / antigüedad / ubicación
    'num_edad_constitucion',
    'num_antiguedad',
    'cnt_meses_siningresos_12m',
    'cnt_meses_sinegresos_12m',
    'cod_ubigeo_cd',
    'cod_sectorista_id',

    # Riesgo / variaciones / casos históricos
    'desc_nivel_rsg_lsb_tot',
    'flg_casos_hist',
    'flg_vrcn_abonos_5m_1m',
    'flg_vrcn_efe_cargos_5m_1m',
    'cnt_ro_debajo_umbral',

    # Perfil económico
    'mto_fact_declarado_sunat',
    'avg_cp_men_ing_12m',
    'avg_cpmenegr_12m',
    'max_mto_cpmening_12m',
    'max_mto_cpegrmen_12m',

    # Exterior
    'flg_al_ext_12m',
    'flg_del_ext_12m',
    'cnt_trx_sinenv_alext_12m',
    'cnt_trx_al_ext_1000_12m',
    'mto_al_ext_12m',
    'mto_del_ext_12m',

    # Reputacional / antecedentes / PEP
    'flg_pep',
    'cod_rsg_pep',
    'flg_activo_pep',
    'cnt_noticias',
    'flg_ros_12m',
    'flg_alerta_12m',
    'cnt_alerta_hist',
    'cnt_ros_hist',

    # KYC
    'flg_kyc_12m',
    'flg_kyc_hist',
    'cnt_kyc_hist',

    # Ratios / features de aceleración y concentración
    'rat_abonos_1m_vs_6m',
    'rat_cargos_1m_vs_6m',
    'rat_egr_princ_contraparte_12m',
    'rat_ing_princ_contraparte_12m',
    'rat_cntros_x_cnttrxegr_3m',
    'rat_alertas_antiguedad_1m',
    'rat_ing_tot_x_factura_6m',
    'rat_pastot_x_ingtot_6m',
    'rat_egr_ext_x_egr_tot_12m',
    'rat_ing_ext_x_ing_tot_12m',
    'gap_riesgo_pep_lsb',

    # 360 cliente
    'edad',
    'ingreso_bruto',

    # RCC
    'prom_lin_tc_rccsf_06m',
    'lin_tcrrstsf03m',
    'prm_usotcrrstsf03m',
    'prm_lintcrallsf12m',
    'prm_lintcrrstsf03m',
    'cre_saltot_tc_rccsf_m02',
    'prom_salvig_entprinc_tc_rccsf_03m',
    'cre_salvig_tc_rccsf_m02',
    'ind_max_salvig_tc_rccsf_06m',
    'var_usotcrrstsf03m',
    'ctd_prod_rccsf_m01',
    'ind_min_salvig_tc_rccsf_06m',
    'lintot_tc_rccsf_03m',
    'prom_salvig_pp_rccsf_06m',
    'salvig_pp_rccsf_06m',
    'cre_pct_salvig_tc_rccsf_m03',
    'prom_salvig_tc_rccsf_06m',
    'cre_lin_tc_rccsf_m02',
    'var_lintcrrstsf03m',

    # Riesgo (perfil de riesgo)
    'ing_brt',
    'ind_lin_ing_tcr_ibk',
    'cem',

    # RCC agg02
    'sow_lnep1tcrallsfm01',
    'sow_svep1actallsfm01',

    # Campañas
    'ctd_camptot06m',
    'prm_camptot06m',
    'max_camptot06m',
    'min_camptot06m',
    'rec_camptot06m',

    # Canales ATM
    'atm_monto',
    'atm_frec',
    'atm_recen',
    'atm_trx_prom',
    'atm_trx_ret_prom',
    'atm_trx_dep',
    'atm_trx_dep_prom',
    'atm_trx_nmon_prom',
    'atm_trx_nmon_prom2',
    'atm_trx_nmon_min',
    'atm_trx_con',
    'atm_trx_con_prom',

    # Canales BPI
    'bpi_monto',
    'bpi_trx_prom',
    'bpi_trx_prom2',
    'bpi_trx_nmon_prom',
    'bpi_trx_nmon_prom2',
    'bpi_trx_mon_prom',
    'bpi_trx_mon_prom2',
    'bpi_trx_mon_min',
    'bpi_trx_mon_max',
    'bpi_recen_mon',
    'bpi_recen_nmon',
]

# ── Filtrar periodos ─────────────────────────────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
print(f"✓ Registros filtrados: {df.shape[0]:,}  |  Target rate: {df[TARGET_COL].mean():.4f}")

# ── Validar columnas existentes ──────────────────────────────────────────────
variables     = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas en el DataFrame: {no_existentes}")
print(f"✓ Variables a graficar: {len(variables)}")

# ── Función de análisis bivariado ────────────────────────────────────────────
def plot_bivariado(df, col, target, n_bins, output_dir):
    df_tmp = df[[col, target]].copy()
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

    try:
        df_tmp["bin"] = pd.qcut(df_tmp[col], q=n_bins, duplicates="drop")
    except Exception:
        df_tmp["bin"] = pd.cut(df_tmp[col], bins=n_bins)

    resumen = (
        df_tmp.groupby("bin", observed=True)[target]
        .agg(count="count", target_rate="mean")
        .reset_index()
    )
    resumen["bin_str"] = resumen["bin"].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.bar(resumen["bin_str"], resumen["count"], color="#4C72B0", alpha=0.7, label="Registros")
    ax1.set_xlabel(col, fontsize=11)
    ax1.set_ylabel("Registros", fontsize=11, color="#4C72B0")
    ax1.tick_params(axis="x", rotation=45, labelsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    ax2 = ax1.twinx()
    ax2.plot(resumen["bin_str"], resumen["target_rate"], color="#DD4949",
             marker="o", linewidth=2, label="Target rate")
    ax2.set_ylabel("Target rate", fontsize=11, color="#DD4949")
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    plt.title(f"Bivariado: {col}  |  periodos {PERIODOS[0]}–{PERIODOS[-1]}", fontsize=13)
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"biv_{col}.png")
    plt.savefig(filepath, dpi=120, bbox_inches="tight")
    plt.close()
    return filepath

# ── Loop principal ───────────────────────────────────────────────────────────
guardados = []
errores   = []

for i, col in enumerate(variables, 1):
    try:
        fp = plot_bivariado(df, col, TARGET_COL, N_BINS, OUTPUT_DIR)
        guardados.append(fp)
        print(f"  [{i}/{len(variables)}] {col} ✓")
    except Exception as e:
        errores.append((col, str(e)))

print(f"\n✅ Gráficos guardados: {len(guardados)}")
print(f"⚠  Errores:           {len(errores)}")
if errores:
    for col, err in errores:
        print(f"   → {col}: {err}")
print(f"\n📁 Carpeta: {OUTPUT_DIR}")


✓ Registros filtrados: 1,795,037  |  Target rate: 0.0002
✓ Variables a graficar: 109
  [1/109] mto_pas_soles ✓
  [2/109] imp_trx_abonosefect_6m ✓
  [3/109] imp_trx_cargosefe_6m ✓
  [4/109] avg_trx_cargostot_3m ✓
  [5/109] max_trx_abonos_3m ✓
  [6/109] cnt_trx_cargostot_3m ✓
  [7/109] cnt_trx_abonospromtot_3m ✓
  [8/109] rat_trx_abonosefectot_1m ✓
  [9/109] rat_trx_abonosefectot_3m ✓
  [10/109] rat_trx_abonosefectot_9m ✓
  [11/109] rat_mntcrgsefetot_1m ✓
  [12/109] num_edad_constitucion ✓
  [13/109] num_antiguedad ✓
  [14/109] cnt_meses_siningresos_12m ✓
  [15/109] cnt_meses_sinegresos_12m ✓
  [16/109] cod_ubigeo_cd ✓
  [17/109] cod_sectorista_id ✓
  [18/109] desc_nivel_rsg_lsb_tot ✓
  [19/109] flg_casos_hist ✓
  [20/109] flg_vrcn_abonos_5m_1m ✓
  [21/109] flg_vrcn_efe_cargos_5m_1m ✓
  [22/109] cnt_ro_debajo_umbral ✓
  [23/109] mto_fact_declarado_sunat ✓
  [24/109] avg_cp_men_ing_12m ✓
  [25/109] avg_cpmenegr_12m ✓
  [26/109] max_mto_cpmening_12m ✓
  [27/109] max_mto_cpegrmen_12m ✓
  [2

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
#import seaborn as sns
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL  = "target_m"
PERIODO_COL = "cod_mes"
PERIODOS    = [202501, 202502, 202503, 202504, 202505, 202506, 202507]
OUTPUT_DIR  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarollo\eda_graficos"
N_BINS      = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

VARIABLES_SELECCIONADAS = [
    # Transaccional / pasivo
    'mto_pas_soles',
    'imp_trx_abonosefect_6m',
    'imp_trx_cargosefe_6m',
    'avg_trx_cargostot_3m',
    'max_trx_abonos_3m',
    'cnt_trx_cargostot_3m',
    'cnt_trx_abonospromtot_3m',
    'rat_trx_abonosefectot_1m',
    'rat_trx_abonosefectot_3m',
    'rat_trx_abonosefectot_9m',
    'rat_mntcrgsefetot_1m',

    # Demográficas / antigüedad / ubicación
    'num_edad_constitucion',
    'num_antiguedad',
    'cnt_meses_siningresos_12m',
    'cnt_meses_sinegresos_12m',
    'cod_ubigeo_cd',
    'cod_sectorista_id',

    # Riesgo / variaciones / casos históricos
    'desc_nivel_rsg_lsb_tot',
    'flg_casos_hist',
    'flg_vrcn_abonos_5m_1m',
    'flg_vrcn_efe_cargos_5m_1m',
    'cnt_ro_debajo_umbral',

    # Perfil económico
    'mto_fact_declarado_sunat',
    'avg_cp_men_ing_12m',
    'avg_cpmenegr_12m',
    'max_mto_cpmening_12m',
    'max_mto_cpegrmen_12m',

    # Exterior
    'flg_al_ext_12m',
    'flg_del_ext_12m',
    'cnt_trx_sinenv_alext_12m',
    'cnt_trx_al_ext_1000_12m',
    'mto_al_ext_12m',
    'mto_del_ext_12m',

    # Reputacional / antecedentes / PEP
    'flg_pep',
    'cod_rsg_pep',
    'flg_activo_pep',
    'cnt_noticias',
    'flg_ros_12m',
    'flg_alerta_12m',
    'cnt_alerta_hist',
    'cnt_ros_hist',

    # KYC
    'flg_kyc_12m',
    'flg_kyc_hist',
    'cnt_kyc_hist',

    # Ratios / features de aceleración y concentración
    'rat_abonos_1m_vs_6m',
    'rat_cargos_1m_vs_6m',
    'rat_egr_princ_contraparte_12m',
    'rat_ing_princ_contraparte_12m',
    'rat_cntros_x_cnttrxegr_3m',
    'rat_alertas_antiguedad_1m',
    'rat_ing_tot_x_factura_6m',
    'rat_pastot_x_ingtot_6m',
    'rat_egr_ext_x_egr_tot_12m',
    'rat_ing_ext_x_ing_tot_12m',
    'gap_riesgo_pep_lsb',

    # 360 cliente
    'edad',
    'ingreso_bruto',

    # RCC
    'prom_lin_tc_rccsf_06m',
    'lin_tcrrstsf03m',
    'prm_usotcrrstsf03m',
    'prm_lintcrallsf12m',
    'prm_lintcrrstsf03m',
    'cre_saltot_tc_rccsf_m02',
    'prom_salvig_entprinc_tc_rccsf_03m',
    'cre_salvig_tc_rccsf_m02',
    'ind_max_salvig_tc_rccsf_06m',
    'var_usotcrrstsf03m',
    'ctd_prod_rccsf_m01',
    'ind_min_salvig_tc_rccsf_06m',
    'lintot_tc_rccsf_03m',
    'prom_salvig_pp_rccsf_06m',
    'salvig_pp_rccsf_06m',
    'cre_pct_salvig_tc_rccsf_m03',
    'prom_salvig_tc_rccsf_06m',
    'cre_lin_tc_rccsf_m02',
    'var_lintcrrstsf03m',

    # Riesgo (perfil de riesgo)
    'ing_brt',
    'ind_lin_ing_tcr_ibk',
    'cem',

    # RCC agg02
    'sow_lnep1tcrallsfm01',
    'sow_svep1actallsfm01',

    # Campañas
    'ctd_camptot06m',
    'prm_camptot06m',
    'max_camptot06m',
    'min_camptot06m',
    'rec_camptot06m',

    # Canales ATM
    'atm_monto',
    'atm_frec',
    'atm_recen',
    'atm_trx_prom',
    'atm_trx_ret_prom',
    'atm_trx_dep',
    'atm_trx_dep_prom',
    'atm_trx_nmon_prom',
    'atm_trx_nmon_prom2',
    'atm_trx_nmon_min',
    'atm_trx_con',
    'atm_trx_con_prom',

    # Canales BPI
    'bpi_monto',
    'bpi_trx_prom',
    'bpi_trx_prom2',
    'bpi_trx_nmon_prom',
    'bpi_trx_nmon_prom2',
    'bpi_trx_mon_prom',
    'bpi_trx_mon_prom2',
    'bpi_trx_mon_min',
    'bpi_trx_mon_max',
    'bpi_recen_mon',
    'bpi_recen_nmon',
]

# ── Filtrar periodos y columnas existentes ───────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
variables = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas: {no_existentes}")

# Convertir a numérico
for col in variables:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df_eda = df[variables + [TARGET_COL]].copy()

print(f"✓ Shape EDA: {df_eda.shape}")
print(f"✓ Target rate: {df_eda[TARGET_COL].mean():.4f}")


# ════════════════════════════════════════════════════════════════════════════
# 1. RESUMEN ESTADÍSTICO
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("1. RESUMEN ESTADÍSTICO")
print("="*60)

stats = df_eda[variables].describe().T
stats["missing"]     = df_eda[variables].isnull().sum()
stats["missing_pct"] = (df_eda[variables].isnull().mean() * 100).round(2)
stats["zeros_pct"]   = ((df_eda[variables] == 0).mean() * 100).round(2)
stats["skewness"]    = df_eda[variables].skew().round(3)
stats["kurtosis"]    = df_eda[variables].kurt().round(3)

display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max",
               "missing_pct", "zeros_pct", "skewness", "kurtosis"]])

stats.to_csv(os.path.join(OUTPUT_DIR, "01_resumen_estadistico.csv"))
print(f"✓ Guardado: 01_resumen_estadistico.csv")


# ════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("2. MISSING VALUES")
print("="*60)

missing = df_eda[variables].isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

if missing.empty:
    print("✓ No hay missing values")
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing) * 0.35)))
    missing.plot(kind="barh", ax=ax, color="#E07B54")
    ax.set_xlabel("% Missing", fontsize=11)
    ax.set_title("Missing Values por Variable (%)", fontsize=13)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    for i, v in enumerate(missing):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_missing_values.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"✓ Guardado: 02_missing_values.png")
    display(missing.to_frame("missing_pct"))


# ════════════════════════════════════════════════════════════════════════════
# 3. DISTRIBUCIONES (histogramas)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("3. DISTRIBUCIONES")
print("="*60)

n_cols  = 4
n_rows  = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    data = df_eda[col].dropna()
    axes[i].hist(data, bins=40, color="#4C72B0", alpha=0.75, edgecolor="white")
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.1f}")
    ))

# Ocultar ejes vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuciones de Variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_distribuciones.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 03_distribuciones.png")


# ════════════════════════════════════════════════════════════════════════════
# 4. CORRELACIÓN CON EL TARGET
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("4. CORRELACIÓN CON EL TARGET")
print("="*60)

corr_target = (
    df_eda[variables + [TARGET_COL]]
    .corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_target) * 0.35)))
colors = ["#DD4949" if v > 0 else "#4C72B0" for v in corr_target]
corr_target.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson", fontsize=11)
ax.set_title(f"Correlación de Variables con '{TARGET_COL}'", fontsize=13)
for i, v in enumerate(corr_target):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_correlacion_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 04_correlacion_target.png")

display(corr_target.to_frame("corr_con_target").style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
corr_target.to_frame("corr_con_target").to_csv(os.path.join(OUTPUT_DIR, "04_correlacion_target.csv"))


# ════════════════════════════════════════════════════════════════════════════
# 5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES (sin seaborn)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES")
print("="*60)

corr_matrix = df_eda[variables].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_plot = corr_matrix.copy()
corr_plot[mask] = np.nan  # ocultar triángulo superior

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(corr_plot, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.6)

# Etiquetas
ax.set_xticks(range(len(variables)))
ax.set_yticks(range(len(variables)))
ax.set_xticklabels(variables, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(variables, fontsize=8)

# Anotaciones numéricas
for i in range(len(variables)):
    for j in range(len(variables)):
        val = corr_plot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=5.5, color="black" if abs(val) < 0.7 else "white")

ax.set_title("Matriz de Correlación entre Variables", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_matriz_correlacion.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 05_matriz_correlacion.png")

# Pares con alta correlación (> 0.7)
corr_upper = corr_matrix.where(mask == False).stack().reset_index()
corr_upper.columns = ["var1", "var2", "correlacion"]
corr_upper = corr_upper[corr_upper["var1"] != corr_upper["var2"]]
alta_corr  = corr_upper[corr_upper["correlacion"].abs() > 0.7].sort_values("correlacion", key=abs, ascending=False)

print(f"\n⚠ Pares con correlación > 0.7:")
display(alta_corr)
alta_corr.to_csv(os.path.join(OUTPUT_DIR, "05_alta_correlacion.csv"), index=False)


# ════════════════════════════════════════════════════════════════════════════
# 6. BOXPLOT TARGET=0 vs TARGET=1
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("6. DISTRIBUCIÓN POR TARGET (0 vs 1)")
print("="*60)

n_cols = 4
n_rows = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    g0 = df_eda[df_eda[TARGET_COL] == 0][col].dropna()
    g1 = df_eda[df_eda[TARGET_COL] == 1][col].dropna()
    axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
                    patch_artist=True,
                    boxprops=dict(facecolor="#4C72B0", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2))
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución por Target (0 vs 1)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_boxplot_por_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 06_boxplot_por_target.png")


# ════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : {len(variables):<25}║
║  Registros            : {df_eda.shape[0]:<25,}║
║  Target rate          : {df_eda[TARGET_COL].mean():<25.4f}║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")


✓ Shape EDA: (1795037, 110)
✓ Target rate: 0.0002

1. RESUMEN ESTADÍSTICO


,count,mean,std,min,25%,50%,75%,max,missing_pct,zeros_pct,skewness,kurtosis
mto_pas_soles,1795037.00,47914.84,291963.12,-97121.37,1.56,641.24,22818.08,80797003.90,0.00,15.03,124.73,27909.73
imp_trx_abonosefect_6m,1312428.00,12400.37,86196.80,0.00,0.00,0.00,3902.42,27452815.42,26.89,43.82,95.52,20666.62
imp_trx_cargosefe_6m,1312428.00,12926.04,120428.03,0.00,0.00,221.00,6036.05,39708735.50,26.89,33.30,144.04,31856.63
avg_trx_cargostot_3m,1795037.00,0.71,1.99,0.00,0.00,0.00,0.33,182.33,0.00,68.40,7.42,180.12
max_trx_abonos_3m,1392825.00,29767.62,187399.04,0.00,20.00,4580.81,20811.13,57502340.37,22.41,13.42,116.50,25930.31
...,...,...,...,...,...,...,...,...,...,...,...,...
bpi_trx_mon_prom2,1378020.00,5.13,11.89,0.00,0.00,1.67,5.89,2424.09,23.23,29.36,41.47,7185.68
bpi_trx_mon_min,1378020.00,1.12,2.87,0.00,0.00,0.00,1.00,399.00,23.23,61.71,10.30,427.25
bpi_trx_mon_max,1378020.00,9.45,58.66,0.00,0.00,3.00,11.00,26543.00,23.23,38.24,403.16,182209.79
bpi_recen_mon,1378020.00,0.56,1.72,0.00,0.00,0.00,0.00,11.00,23.23,85.02,3.62,13.16


✓ Guardado: 01_resumen_estadistico.csv

2. MISSING VALUES
✓ Guardado: 02_missing_values.png


,missing_pct
num_edad_constitucion,100.00
flg_activo_pep,100.00
cnt_kyc_hist,100.00
mto_fact_declarado_sunat,100.00
desc_nivel_rsg_lsb_tot,100.00
...,...
ind_lin_ing_tcr_ibk,18.19
ing_brt,18.19
cem,18.19
cod_rsg_pep,5.99



3. DISTRIBUCIONES
✓ Guardado: 03_distribuciones.png

4. CORRELACIÓN CON EL TARGET
✓ Guardado: 04_correlacion_target.png


...


5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES
✓ Guardado: 05_matriz_correlacion.png

⚠ Pares con correlación > 0.7:


,var1,var2,correlacion
3398,atm_trx_dep_prom,atm_trx_dep,1.00
3823,atm_trx_con_prom,atm_trx_con,1.00
1319,prm_lintcrrstsf03m,lin_tcrrstsf03m,1.00
2623,prm_camptot06m,ctd_camptot06m,1.00
1754,lintot_tc_rccsf_03m,prom_lin_tc_rccsf_06m,0.99
4460,bpi_trx_mon_prom2,bpi_trx_mon_prom,0.99
377,cnt_trx_al_ext_1000_12m,cnt_trx_sinenv_alext_12m,0.99
1268,prm_lintcrallsf12m,prom_lin_tc_rccsf_06m,0.99
1479,cre_salvig_tc_rccsf_m02,cre_saltot_tc_rccsf_m02,0.98
1757,lintot_tc_rccsf_03m,prm_lintcrallsf12m,0.98



6. DISTRIBUCIÓN POR TARGET (0 vs 1)


C:\Users\b46637\AppData\Local\Temp\ipykernel_30324\3524288124.py:350: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_30324\3524288124.py:350: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_30324\3524288124.py:350: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_30324\3524288124.py:350: MatplotlibDeprecationWarning: 

✓ Guardado: 06_boxplot_por_target.png

╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : 109                      ║
║  Registros            : 1,795,037                ║
║  Target rate          : 0.0002                   ║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarollo\eda_graficos

In [14]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# SELECCIÓN DE VARIABLES PARA EL MODELO
# Criterios:
#   1. Missing % < MAX_MISSING_PCT  (hasta 99% de nulos permitido)
#   2. Sin multicolinealidad: |corr entre variables| <= CORR_THRESHOLD
#      → se elimina la de MENOR correlación con el target
# ═══════════════════════════════════════════════════════════════════════════

MAX_MISSING_PCT = 99.0   # elimina solo variables con 100% nulos
CORR_THRESHOLD  = 0.70   # correlación máxima permitida entre variables

print("=" * 65)
print("SELECCIÓN DE VARIABLES PARA EL MODELO")
print("=" * 65)
print(f"  Umbral missing   : < {MAX_MISSING_PCT}%  (se eliminan solo las que tienen 100% nulos)")
print(f"  Umbral corr vars : <= {CORR_THRESHOLD}")
print(f"  Variables iniciales: {len(variables)}\n")

# ── 1. Filtro por missing ────────────────────────────────────────────────────
missing_pct = df_eda[variables].isnull().mean() * 100
vars_ok_missing   = missing_pct[missing_pct < MAX_MISSING_PCT].index.tolist()
vars_drop_missing = missing_pct[missing_pct >= MAX_MISSING_PCT].index.tolist()

print(f"── Filtro 1: MISSING >= {MAX_MISSING_PCT}%  →  eliminadas {len(vars_drop_missing)}")
for v in vars_drop_missing:
    print(f"   ✗ {v:<45}  ({missing_pct[v]:.1f}% nulos)")
print(f"   Quedan: {len(vars_ok_missing)} variables\n")

# ── 2. Correlación con el target ─────────────────────────────────────────────
corr_con_target = (
    df_eda[vars_ok_missing + [TARGET_COL]]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL, errors="ignore")
    .abs()
    .fillna(0)
)

# ── 3. Filtro multicolinealidad (greedy, conserva la más predictiva) ──────────
corr_matrix = df_eda[vars_ok_missing].corr(numeric_only=True).abs()
vars_num    = corr_matrix.columns.tolist()

eliminadas_corr = []
seleccionadas   = sorted(vars_num, key=lambda v: corr_con_target.get(v, 0), reverse=True)

i = 0
while i < len(seleccionadas):
    v1 = seleccionadas[i]
    j  = i + 1
    while j < len(seleccionadas):
        v2 = seleccionadas[j]
        if corr_matrix.loc[v1, v2] > CORR_THRESHOLD:
            eliminadas_corr.append((v2, v1, round(corr_matrix.loc[v1, v2], 3)))
            seleccionadas.pop(j)
        else:
            j += 1
    i += 1

vars_no_num = [v for v in vars_ok_missing if v not in vars_num]

print(f"── Filtro 2: CORRELACIÓN > {CORR_THRESHOLD}  →  eliminadas {len(eliminadas_corr)}")
for v_elim, v_keep, corr_val in eliminadas_corr:
    print(f"   ✗ {v_elim:<45}  (corr={corr_val:.3f} con '{v_keep}')")
print(f"   Quedan: {len(seleccionadas)} variables numéricas\n")

if vars_no_num:
    print(f"── Variables categóricas/string (sin filtro corr): {vars_no_num}\n")

# ── Lista final ───────────────────────────────────────────────────────────────
VARIABLES_MODELO = seleccionadas + vars_no_num

# ── Tabla resumen con selector interactivo ────────────────────────────────────
df_selector = pd.DataFrame({
    "variable"    : VARIABLES_MODELO,
    "missing_pct" : [round(missing_pct.get(v, 0), 2) for v in VARIABLES_MODELO],
    "corr_target" : [round(corr_con_target.get(v, float("nan")), 6) for v in VARIABLES_MODELO],
    "tipo"        : ["numérica" if v in vars_num else "categórica" for v in VARIABLES_MODELO],
    "incluir"     : [True] * len(VARIABLES_MODELO),   # puedes editar aquí para excluir manualmente
}).sort_values("corr_target", ascending=False).reset_index(drop=True)

print("=" * 65)
print(f"✅ VARIABLES SELECCIONADAS: {len(VARIABLES_MODELO)}")
print("=" * 65)

display(
    df_selector.style
    .background_gradient(subset=["corr_target"], cmap="Greens")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .format({"missing_pct": "{:.1f}%", "corr_target": "{:.6f}"})
    .set_properties(**{"font-size": "11px"})
)

# ── Guardar tabla de selección ────────────────────────────────────────────────
OUTPUT_SEL = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarrollo\variables_modelo.csv"
os.makedirs(os.path.dirname(OUTPUT_SEL), exist_ok=True)
df_selector.to_csv(OUTPUT_SEL, index=False)
print(f"\n✓ Tabla guardada en: {OUTPUT_SEL}")

# ── Aplicar columna 'incluir' para obtener lista final editable ───────────────
VARIABLES_MODELO_FINAL = df_selector.loc[df_selector["incluir"] == True, "variable"].tolist()

print(f"\n📋 Lista final ({len(VARIABLES_MODELO_FINAL)} variables):")
print("VARIABLES_MODELO_FINAL = [")
for v in VARIABLES_MODELO_FINAL:
    print(f"    '{v}',")
print("]")

SELECCIÓN DE VARIABLES PARA EL MODELO
  Umbral missing   : < 99.0%  (se eliminan solo las que tienen 100% nulos)
  Umbral corr vars : <= 0.7
  Variables iniciales: 109

── Filtro 1: MISSING >= 99.0%  →  eliminadas 7
   ✗ num_edad_constitucion                          (100.0% nulos)
   ✗ desc_nivel_rsg_lsb_tot                         (100.0% nulos)
   ✗ mto_fact_declarado_sunat                       (100.0% nulos)
   ✗ flg_activo_pep                                 (100.0% nulos)
   ✗ cnt_kyc_hist                                   (100.0% nulos)
   ✗ rat_ing_tot_x_factura_6m                       (100.0% nulos)
   ✗ gap_riesgo_pep_lsb                             (100.0% nulos)
   Quedan: 102 variables

── Filtro 2: CORRELACIÓN > 0.7  →  eliminadas 27
   ✗ cnt_alerta_hist                                (corr=0.776 con 'rat_alertas_antiguedad_1m')
   ✗ max_mto_cpmening_12m                           (corr=0.954 con 'avg_cp_men_ing_12m')
   ✗ bpi_trx_prom                                   (

,variable,missing_pct,corr_target,tipo,incluir
0,mto_del_ext_12m,96.9%,0.042948,numérica,True
1,max_trx_abonos_3m,22.4%,0.041175,numérica,True
2,cnt_ros_hist,90.5%,0.040598,numérica,True
3,imp_trx_abonosefect_6m,26.9%,0.039569,numérica,True
4,rat_alertas_antiguedad_1m,91.2%,0.035716,numérica,True
5,imp_trx_cargosefe_6m,26.9%,0.035035,numérica,True
6,flg_ros_12m,0.0%,0.034414,numérica,True
7,flg_alerta_12m,0.0%,0.031683,numérica,True
8,rat_cargos_1m_vs_6m,60.2%,0.031606,numérica,True
9,bpi_monto,23.2%,0.030623,numérica,True



✓ Tabla guardada en: C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarrollo\variables_modelo.csv

📋 Lista final (75 variables):
VARIABLES_MODELO_FINAL = [
    'mto_del_ext_12m',
    'max_trx_abonos_3m',
    'cnt_ros_hist',
    'imp_trx_abonosefect_6m',
    'rat_alertas_antiguedad_1m',
    'imp_trx_cargosefe_6m',
    'flg_ros_12m',
    'flg_alerta_12m',
    'rat_cargos_1m_vs_6m',
    'bpi_monto',
    'avg_cp_men_ing_12m',
    'bpi_trx_nmon_prom',
    'cnt_trx_cargostot_3m',
    'atm_monto',
    'atm_trx_dep',
    'bpi_trx_mon_prom2',
    'mto_pas_soles',
    'cnt_trx_abonospromtot_3m',
    'mto_al_ext_12m',
    'cnt_meses_sinegresos_12m',
    'atm_trx_nmon_prom2',
    'avg_trx_cargostot_3m',
    'rat_trx_abonosefectot_3m',
    'atm_trx_nmon_min',
    'bpi_trx_mon_min',
    'atm_frec',
    'flg_al_ext_12m',
    'cnt_trx_al_ext_1000_12m',
    'rat_cntros_x_cnttrxegr_3m',
    'ing_brt',
    'bpi_trx_mon_max',
    'flg_pep',
    'cod_rsg_pep',


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
import os

# ═══════════════════════════════════════════════════════════════════════════
# SELECTOR POR IMPORTANCIA ACUMULADA (LightGBM)
# Conserva las variables que acumulan el UMBRAL_IMPORTANCIA de la importancia
# total del modelo.
# ═══════════════════════════════════════════════════════════════════════════

UMBRAL_IMPORTANCIA = 0.99   # 99% de importancia acumulada
OUTPUT_DIR_SEL     = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarrollo\eda_graficos_train"
os.makedirs(OUTPUT_DIR_SEL, exist_ok=True)

# ── Preparar datos de entrenamiento ─────────────────────────────────────────
# Usamos VARIABLES_MODELO_FINAL (salida de la celda anterior)
# Solo columnas numéricas (LightGBM acepta NaN pero no strings sin encode)
vars_lgb = [v for v in VARIABLES_MODELO_FINAL
            if v in df_eda.select_dtypes(include=[np.number]).columns]

print(f"Variables numéricas para LightGBM : {len(vars_lgb)}")
print(f"Variables categóricas excluidas   : "
      f"{[v for v in VARIABLES_MODELO_FINAL if v not in vars_lgb]}\n")

X = df_eda[vars_lgb].copy()
y = df_eda[TARGET_COL].copy()

# Eliminar filas donde el target es NaN
mask_valid = y.notna()
X, y = X[mask_valid], y[mask_valid]

print(f"✓ Shape para entrenamiento: {X.shape}  |  Target rate: {y.mean():.5f}")

# ── Entrenar LightGBM rápido (solo para importancia) ────────────────────────
scale_pos = (y == 0).sum() / max((y == 1).sum(), 1)

params = {
    "objective"        : "binary",
    "metric"           : "auc",
    "n_estimators"     : 300,
    "learning_rate"    : 0.05,
    "num_leaves"       : 63,
    "min_child_samples": 50,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "scale_pos_weight" : scale_pos,
    "n_jobs"           : -1,
    "random_state"     : 42,
    "verbose"          : -1,
}

print("\n⏳ Entrenando LightGBM para calcular importancia de variables...")
model = lgb.LGBMClassifier(**params)
model.fit(X, y)
print("✓ Entrenamiento completado")

# ── Importancia acumulada ───────────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=vars_lgb)
importances = importances.sort_values(ascending=False)
importancias_norm = importances / importances.sum()
importancia_acum  = importancias_norm.cumsum()

# Variables que acumulan hasta UMBRAL_IMPORTANCIA
vars_99 = importancia_acum[importancia_acum <= UMBRAL_IMPORTANCIA].index.tolist()

# Asegurarse de incluir la variable que cruza el umbral (si quedaron justo por debajo)
if len(vars_99) < len(vars_lgb):
    siguiente = importancia_acum.index[len(vars_99)]
    vars_99.append(siguiente)

print(f"\n✅ Variables que acumulan {UMBRAL_IMPORTANCIA*100:.0f}% de importancia: {len(vars_99)} / {len(vars_lgb)}")

# ── Tabla resultado ─────────────────────────────────────────────────────────
df_importancia = pd.DataFrame({
    "variable"           : importancias_norm.index,
    "importancia"        : importancias_norm.values,
    "importancia_acum"   : importancia_acum.values,
    "missing_pct"        : [round(missing_pct.get(v, 0), 2) for v in importancias_norm.index],
    "seleccionada"       : [v in vars_99 for v in importancias_norm.index],
}).reset_index(drop=True)

display(
    df_importancia.style
    .background_gradient(subset=["importancia"], cmap="Greens")
    .background_gradient(subset=["importancia_acum"], cmap="Blues")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .apply(lambda col: ["background-color: #d4edda" if v else "" for v in df_importancia["seleccionada"]], axis=0)
    .format({
        "importancia"      : "{:.4%}",
        "importancia_acum" : "{:.2%}",
        "missing_pct"      : "{:.1f}%",
    })
    .set_properties(**{"font-size": "11px"})
)

# ── Gráfico importancia acumulada ───────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(max(14, len(vars_lgb) * 0.25), 6))

bars = ax1.bar(range(len(importancias_norm)), importancias_norm.values,
               color=["#2ecc71" if v in vars_99 else "#bdc3c7" for v in importancias_norm.index],
               alpha=0.85, label="Importancia individual")
ax1.set_xticks(range(len(importancias_norm)))
ax1.set_xticklabels(importancias_norm.index, rotation=90, fontsize=7)
ax1.set_ylabel("Importancia relativa", fontsize=11)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2 = ax1.twinx()
ax2.plot(range(len(importancia_acum)), importancia_acum.values,
         color="#e74c3c", linewidth=2, marker=".", markersize=4, label="Importancia acumulada")
ax2.axhline(UMBRAL_IMPORTANCIA, color="#e74c3c", linestyle="--", linewidth=1,
            label=f"Umbral {UMBRAL_IMPORTANCIA*100:.0f}%")
ax2.set_ylabel("Importancia acumulada", fontsize=11, color="#e74c3c")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right", fontsize=9)

plt.title(f"Importancia de Variables (LightGBM)  |  {len(vars_99)} vars explican {UMBRAL_IMPORTANCIA*100:.0f}% de importancia",
          fontsize=13)
plt.tight_layout()
fp_imp = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.png")
plt.savefig(fp_imp, dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Gráfico guardado: {fp_imp}")

# ── Guardar tabla ───────────────────────────────────────────────────────────
fp_csv = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.csv")
df_importancia.to_csv(fp_csv, index=False)
print(f"✓ Tabla guardada : {fp_csv}")

# ── Lista final para el modelo ──────────────────────────────────────────────
VARIABLES_FINALES_MODELO = vars_99

print(f"\n{'='*65}")
print(f"📋 VARIABLES_FINALES_MODELO ({len(VARIABLES_FINALES_MODELO)} variables)")
print(f"{'='*65}")
print("VARIABLES_FINALES_MODELO = [")
for v in VARIABLES_FINALES_MODELO:
    imp = importancias_norm.get(v, 0)
    print(f"    '{v}',  # importancia={imp:.4%}")
print("]")


Variables numéricas para LightGBM : 75
Variables categóricas excluidas   : []

✓ Shape para entrenamiento: (1795037, 75)  |  Target rate: 0.00024

⏳ Entrenando LightGBM para calcular importancia de variables...
✓ Entrenamiento completado

✅ Variables que acumulan 99% de importancia: 57 / 75


,variable,importancia,importancia_acum,missing_pct,seleccionada
0,rat_alertas_antiguedad_1m,6.0498%,6.05%,91.2%,True
1,cnt_trx_cargostot_3m,4.6263%,10.68%,22.4%,True
2,max_trx_abonos_3m,4.4484%,15.12%,22.4%,True
3,ingreso_bruto,4.2705%,19.40%,23.2%,True
4,rat_cntros_x_cnttrxegr_3m,4.0925%,23.49%,92.1%,True
5,prom_salvig_tc_rccsf_06m,4.0925%,27.58%,21.6%,True
6,rat_pastot_x_ingtot_6m,4.0925%,31.67%,30.9%,True
7,mto_pas_soles,3.5587%,35.23%,0.0%,True
8,cod_ubigeo_cd,3.3808%,38.61%,0.0%,True
9,bpi_monto,3.3808%,41.99%,23.2%,True


✓ Gráfico guardado: C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarrollo\eda_graficos_train\07_importancia_variables.png
✓ Tabla guardada : C:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Renta_Alta_regulado\Desarrollo\eda_graficos_train\07_importancia_variables.csv

📋 VARIABLES_FINALES_MODELO (57 variables)
VARIABLES_FINALES_MODELO = [
    'rat_alertas_antiguedad_1m',  # importancia=6.0498%
    'cnt_trx_cargostot_3m',  # importancia=4.6263%
    'max_trx_abonos_3m',  # importancia=4.4484%
    'ingreso_bruto',  # importancia=4.2705%
    'rat_cntros_x_cnttrxegr_3m',  # importancia=4.0925%
    'prom_salvig_tc_rccsf_06m',  # importancia=4.0925%
    'rat_pastot_x_ingtot_6m',  # importancia=4.0925%
    'mto_pas_soles',  # importancia=3.5587%
    'cod_ubigeo_cd',  # importancia=3.3808%
    'bpi_monto',  # importancia=3.3808%
    'edad',  # importancia=3.2028%
    'rat_abonos_1m_vs_6m',  # importancia=3.0249%
    'imp_trx_ca

In [17]:
VARIABLES_FINALES_MODELO

['rat_alertas_antiguedad_1m',
 'cnt_trx_cargostot_3m',
 'max_trx_abonos_3m',
 'ingreso_bruto',
 'rat_cntros_x_cnttrxegr_3m',
 'prom_salvig_tc_rccsf_06m',
 'rat_pastot_x_ingtot_6m',
 'mto_pas_soles',
 'cod_ubigeo_cd',
 'bpi_monto',
 'edad',
 'rat_abonos_1m_vs_6m',
 'imp_trx_cargosefe_6m',
 'cod_sectorista_id',
 'atm_monto',
 'imp_trx_abonosefect_6m',
 'max_camptot06m',
 'cnt_trx_abonospromtot_3m',
 'cnt_ros_hist',
 'max_mto_cpegrmen_12m',
 'num_antiguedad',
 'var_usotcrrstsf03m',
 'bpi_trx_nmon_prom',
 'cnt_meses_sinegresos_12m',
 'ind_min_salvig_tc_rccsf_06m',
 'bpi_trx_mon_prom2',
 'lin_tcrrstsf03m',
 'cre_salvig_tc_rccsf_m02',
 'ing_brt',
 'cre_lin_tc_rccsf_m02',
 'sow_lnep1tcrallsfm01',
 'avg_cp_men_ing_12m',
 'ind_max_salvig_tc_rccsf_06m',
 'mto_al_ext_12m',
 'sow_svep1actallsfm01',
 'rat_mntcrgsefetot_1m',
 'rec_camptot06m',
 'rat_cargos_1m_vs_6m',
 'mto_del_ext_12m',
 'cre_pct_salvig_tc_rccsf_m03',
 'rat_trx_abonosefectot_3m',
 'prm_usotcrrstsf03m',
 'atm_recen',
 'flg_del_ext_12